In [ ]:
# =========================
# nb_run_all
# =========================

# ---------- Parameters ----------
period = "2026-03"   # YYYY-MM

# ---------- Imports ----------
from datetime import datetime, timezone
from pyspark.sql import functions as F

# ---------- Run identity ----------
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"[nb_run_all] START period={period} run_id={run_id}")

# ---------- Helpers ----------
def q(s: str) -> str:
    return (s or "").replace("'", "''")

def table_exists(table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False

def audit_orchestrator(status, message):
    spark.sql(f"""
        INSERT INTO run_audit_log
        (run_id, period, dfm_id, files_processed, rows_ingested, parse_errors_count, drift_events_count, status, started_at, completed_at)
        VALUES
        ('{q(run_id)}','{q(period)}','orchestrator',0,0,0,0,'{q(status)}',current_timestamp(),current_timestamp())
    """)
    print(f"[orchestrator] {status}: {message}")

def run_notebook(nb_name, args, timeout_sec=3600):
    print(f"\n--- Running {nb_name} with args={args}")
    try:
        res = mssparkutils.notebook.run(nb_name, timeout_sec, args)
        res_text = (res or "").strip()
        print(f"--- {nb_name} completed with result: {res_text}")
        return "OK", res_text
    except Exception as ex:
        err = f"{type(ex).__name__}: {str(ex)}"
        print(f"--- {nb_name} FAILED: {err}")
        return "FAILED", err

def ensure_stage1_from_canonical(period_value: str, run_id_value: str):
    # Compatibility bridge while DFM adapters still write canonical_holdings.
    if not table_exists("canonical_holdings"):
        print("[bridge] canonical_holdings not found; skipping Stage 1 bridge")
        return

    source_df = spark.table("canonical_holdings").filter(
        (F.col("period") == period_value) & (F.col("run_id") == run_id_value)
    )

    if source_df.rdd.isEmpty():
        print("[bridge] no canonical_holdings rows to bridge for Stage 1")
        return

    stage1 = (
        source_df
        .withColumn("profile_id", F.col("dfm_id"))
        .withColumn(
            "raw_record_json",
            F.to_json(
                F.struct(
                    F.col("policy_id"),
                    F.col("security_id"),
                    F.col("isin"),
                    F.col("other_security_id"),
                    F.col("asset_name"),
                    F.col("holding"),
                    F.col("local_bid_price"),
                    F.col("local_currency"),
                    F.col("fx_rate"),
                    F.col("cash_value_gbp"),
                    F.col("bid_value_gbp"),
                    F.col("accrued_interest_gbp"),
                    F.col("report_date"),
                    F.col("data_quality_flags")
                )
            )
        )
        .withColumn("parse_status", F.lit("ok"))
        .withColumn("parse_error_code", F.lit(None).cast("string"))
        .select(
            "period",
            "run_id",
            "dfm_id",
            "profile_id",
            "source_file",
            "source_sheet",
            "source_row_id",
            "raw_record_json",
            "parse_status",
            "parse_error_code",
            F.current_timestamp().alias("ingested_at")
        )
        .dropDuplicates(["run_id", "dfm_id", "source_row_id"])
    )

    stage1.write.mode("append").saveAsTable("source_dfm_raw")
    print(f"[bridge] appended {stage1.count()} rows to source_dfm_raw")

def ensure_stage2_from_canonical(period_value: str, run_id_value: str):
    # Compatibility bridge while DFM adapters still write canonical_holdings.
    if not table_exists("canonical_holdings"):
        print("[bridge] canonical_holdings not found; skipping Stage 2 bridge")
        return

    source_df = spark.table("canonical_holdings").filter(
        (F.col("period") == period_value) & (F.col("run_id") == run_id_value)
    )

    if source_df.rdd.isEmpty():
        print("[bridge] no canonical_holdings rows to bridge")
        return

    stage2 = (
        source_df
        .withColumn("profile_id", F.col("dfm_id"))
        .withColumn("row_hash", F.sha2(F.concat_ws("|", F.col("run_id"), F.col("dfm_id"), F.col("source_row_id")), 256))
        .withColumn("policyholder_number", F.col("policy_id"))
        .withColumn("security_code", F.col("security_id"))
        .withColumn("sedol", F.lit(None).cast("string"))
        .withColumn("include_flag", F.lit("Include"))
        .withColumn("exclusion_reason_code", F.lit(None).cast("string"))
        .withColumn("identifier_chosen", F.coalesce(F.col("security_id"), F.col("isin")))
        .withColumn(
            "decision_trace_json",
            F.to_json(
                F.struct(
                    F.lit("bridge_from_canonical_holdings").alias("strategy"),
                    F.col("dfm_id").alias("dfm_id"),
                    F.current_timestamp().alias("bridged_at")
                )
            )
        )
        .select(
            "period",
            "run_id",
            "dfm_id",
            "profile_id",
            "source_file",
            "source_row_id",
            "row_hash",
            "policyholder_number",
            "security_code",
            "isin",
            "sedol",
            "other_security_id",
            "id_type",
            "asset_name",
            F.col("holding").cast("decimal(28,8)").alias("holding"),
            F.col("local_bid_price").cast("decimal(28,8)").alias("local_bid_price"),
            "local_currency",
            F.col("fx_rate").cast("decimal(28,8)").alias("fx_rate"),
            F.col("cash_value_gbp").cast("decimal(28,8)").alias("cash_value_gbp"),
            F.col("bid_value_gbp").cast("decimal(28,8)").alias("bid_value_gbp"),
            F.col("accrued_interest_gbp").cast("decimal(28,8)").alias("accrued_interest_gbp"),
            "include_flag",
            "exclusion_reason_code",
            "identifier_chosen",
            "decision_trace_json",
            "data_quality_flags"
        )
        .dropDuplicates(["run_id", "dfm_id", "source_row_id"])
    )

    stage2.write.mode("append").saveAsTable("individual_dfm_consolidated")
    print(f"[bridge] appended {stage2.count()} rows to individual_dfm_consolidated")

def has_blocking_gate_failures(period_value: str, run_id_value: str) -> bool:
    if not table_exists("dq_results"):
        return False
    dq = spark.table("dq_results").filter(
        (F.col("period") == period_value)
        & (F.col("run_id") == run_id_value)
        & (F.lower(F.col("severity")).isin("exception", "stop"))
        & (F.lower(F.col("status")) == "fail")
    )
    return not dq.rdd.isEmpty()

# ---------- Phase 8 & 9 enablement ----------
enable_phase_8 = False  # Set to True to enable TPIR validation + ADS load
enable_phase_9 = False  # Set to True to enable AI workflows

# ---------- Execution plan ----------
ingest_jobs = [
    ("nb_ingest_wh_ireland", "wh_ireland"),
    ("nb_ingest_pershing", "pershing"),
    ("nb_ingest_castlebay", "castlebay"),
    ("nb_ingest_brown_shipley", "brown_shipley"),
]

phase_8_jobs = [
    "nb_tpir_check",
    "nb_ads_load",
]

phase_9_jobs = [
    "nb_fuzzy_resolution",
    "nb_anomaly_detection",
    "nb_exception_triage",
    "nb_narrative_generation",
    "nb_summary",
]

# ---------- Start audit ----------
audit_orchestrator("RUNNING", "Orchestration started")

# ---------- Run ingestion notebooks ----------
ingest_results = []
any_hard_fail = False

for nb, dfm in ingest_jobs:
    status, detail = run_notebook(
        nb_name=nb,
        args={"period": period, "run_id": run_id},
        timeout_sec=3600,
    )
    ingest_results.append((nb, dfm, status, detail))
    if status == "FAILED":
        any_hard_fail = True

ingest_ok_or_partial = [r for r in ingest_results if r[2] != "FAILED"]
if len(ingest_ok_or_partial) == 0:
    audit_orchestrator("FAILED", "All ingest notebooks failed; skipping downstream stages")
    mssparkutils.notebook.exit(f"FAILED|run_id={run_id}|reason=all_ingests_failed")

# ---------- Bridge to Stage 1/2 until all adapters are migrated ----------
ensure_stage1_from_canonical(period, run_id)
ensure_stage2_from_canonical(period, run_id)

# ---------- Run validation stage first ----------
post_results = []
validate_status, validate_detail = run_notebook(
    nb_name="nb_validate",
    args={"period": period, "run_id": run_id},
    timeout_sec=3600,
)
post_results.append(("nb_validate", validate_status, validate_detail))
if validate_status == "FAILED":
    any_hard_fail = True

# ---------- Enforce publish gate for Stage 3 ----------
blocked_by_gate = has_blocking_gate_failures(period, run_id)
if blocked_by_gate:
    print("[gate] Blocking failures detected in dq_results; skipping nb_aggregate and nb_reports")
else:
    for nb in ["nb_aggregate", "nb_reports"]:
        status, detail = run_notebook(
            nb_name=nb,
            args={"period": period, "run_id": run_id},
            timeout_sec=3600,
        )
        post_results.append((nb, status, detail))
        if status == "FAILED":
            any_hard_fail = True
            break

# ---------- Run optional phases ----------
phase_8_results = []
if enable_phase_8:
    print("\n=== PHASE 8 EXECUTION (TPIR + ADS) ===")
    for nb in phase_8_jobs:
        status, detail = run_notebook(nb_name=nb, args={"period": period, "run_id": run_id}, timeout_sec=3600)
        phase_8_results.append((nb, status, detail))
        if status == "FAILED":
            print(f"[nb_run_all] Phase 8 notebook {nb} failed; continuing")
else:
    print("\n[nb_run_all] Phase 8 SKIPPED (enable_phase_8=False)")

phase_9_results = []
if enable_phase_9:
    print("\n=== PHASE 9 EXECUTION (AI WORKFLOWS) ===")
    for nb in phase_9_jobs:
        status, detail = run_notebook(nb_name=nb, args={"period": period, "run_id": run_id}, timeout_sec=3600)
        phase_9_results.append((nb, status, detail))
        if status == "FAILED":
            print(f"[nb_run_all] Phase 9 notebook {nb} failed; continuing")
else:
    print("\n[nb_run_all] Phase 9 SKIPPED (enable_phase_9=False)")

# ---------- Final status ----------
ingest_failed_count = len([r for r in ingest_results if r[2] == "FAILED"])
post_failed_count = len([r for r in post_results if r[1] == "FAILED"])
phase_8_failed_count = len([r for r in phase_8_results if r[1] == "FAILED"])
phase_9_failed_count = len([r for r in phase_9_results if r[1] == "FAILED"])

if blocked_by_gate and not any_hard_fail:
    final_status = "BLOCKED"
elif any_hard_fail:
    final_status = "PARTIAL" if len(ingest_ok_or_partial) > 0 else "FAILED"
else:
    final_status = "OK"

summary = (
    f"run_id={run_id}; "
    f"ingest_failed={ingest_failed_count}/{len(ingest_results)}; "
    f"post_failed={post_failed_count}/{len(post_results)}; "
    f"gate_blocked={blocked_by_gate}; "
    f"phase_8_failed={phase_8_failed_count}/{len(phase_8_results) if enable_phase_8 else 0}; "
    f"phase_9_failed={phase_9_failed_count}/{len(phase_9_results) if enable_phase_9 else 0}"
)

audit_orchestrator(final_status, summary)

print("\n=== RUN SUMMARY ===")
print(summary)
print("\nIngest results:")
for r in ingest_results:
    print(f"  {r[0]}: {r[2]}")
print("Post results:")
for r in post_results:
    print(f"  {r[0]}: {r[1]}")
if phase_8_results:
    print("Phase 8 results:")
    for r in phase_8_results:
        print(f"  {r[0]}: {r[1]}")
if phase_9_results:
    print("Phase 9 results:")
    for r in phase_9_results:
        print(f"  {r[0]}: {r[1]}")

mssparkutils.notebook.exit(f"{final_status}|run_id={run_id}")

StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, 3, Finished, Available, Finished, False)

[nb_run_all] START period=2026-03 run_id=20260302T220333Z
[orchestrator] RUNNING: Orchestration started

--- Running nb_ingest_wh_ireland with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_wh_ireland completed with result: NO_FILES

--- Running nb_ingest_pershing with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_ingest_pershing FAILED: Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362
    330 good = good.withColumn(
    331     "source_row_id",
    332     F.sha2(
   (...)
    344     )
    345 )
    347 # ---------- Canonical projection ----------
    348 canonical = (
    349     good
    350     .withColumn("period", F.lit(period))
    351     .withColumn("run_id", F.lit(run_id))
    352     .withColumn("dfm_id", F.lit(dfm_id))
    353     .withColumn("dfm_name", F.lit(dfm_name))
    354     .withColumn("source_file", F.col("source_file_out"))
    355     .withColumn("source_sheet", F.lit(None).cast("string"))
    356     .wi

--- nb_ingest_castlebay FAILED: Py4JJavaError: An error occurred while calling o5229.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 85f41908-fd38-4b99-b667-d1fb8262c0c6).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- period: string (nullable = true)
-- run_id: string (nullable = true)
-- dfm_id: string (nullable = true)
-- source_file: string (nullable = true)
-- row_number: long (nullable = true)
-- column_name: string (nullable = true)
-- raw_value: string (nullable = true)
-- error_type: string (nullable = true)
-- error_message: string (nullable = true)
-- event_ts: timestamp (nul

--- nb_ingest_brown_shipley FAILED: Py4JJavaError: An error occurred while calling o5239.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].
---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 388
    355 good = good.withColumn(
    356     "source_row_id",
    357     F.sha2(
   (...)
    371     )
    372 )
    374 # ---------- Canonical projection ----------
    375 canonical = (
    376     good
    377     .withColumn("period", F.lit(period))
    378     .withColumn("run_id", F.lit(run_id))
    379     .withColumn("dfm_id", F.lit(dfm_id))
    380     .withColumn("dfm_name", F.lit(dfm_name))
    381     .withColumn("source_file", F.col("source_file_out"))
    382     .withColumn("source_sheet", F.lit(None).cast("string"))
    383   

--- nb_validate completed with result: NO_DATA

--- Running nb_aggregate with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_aggregate completed with result: NO_DATA

--- Running nb_reports with args={'period': '2026-03', 'run_id': '20260302T220333Z'}


--- nb_reports completed with result: NO_DATA
[orchestrator] PARTIAL: run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3

=== RUN SUMMARY ===
run_id=20260302T220333Z; ingest_failed=3/4; post_failed=0/3
Ingest results:
('nb_ingest_wh_ireland', 'wh_ireland', 'OK', 'NO_FILES')
('nb_ingest_pershing', 'pershing', 'FAILED', 'Py4JJavaError: An error occurred while calling o5219.throwExceptionIfHave.\n: com.microsoft.spark.notebook.msutils.NotebookExecutionException: [AMBIGUOUS_REFERENCE] Reference `source_file` is ambiguous, could be: [`source_file`, `source_file`].\n---------------------------------------------------------------------------AnalysisException                         Traceback (most recent call last)Cell In[4], line 362\n    330 good = good.withColumn(\n    331     "source_row_id",\n    332     F.sha2(\n   (...)\n    344     )\n    345 )\n    347 # ---------- Canonical projection ----------\n    348 canonical = (\n    349     good\n    350     .withColumn("period", F.l

In [ ]:
# 1) audit rows for this run
spark.sql(f"""
SELECT run_id, period, dfm_id, status, files_processed, rows_ingested, parse_errors_count, started_at, completed_at
FROM run_audit_log
WHERE run_id = '{run_id}'
ORDER BY dfm_id, completed_at
""").show(truncate=False)

# 2) stage 1 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM source_dfm_raw
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

# 3) stage 2 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM individual_dfm_consolidated
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

# 4) stage 3 row counts by DFM
spark.sql(f"""
SELECT dfm_id, COUNT(*) AS rows
FROM aggregated_dfms_consolidated
WHERE run_id = '{run_id}'
GROUP BY dfm_id
ORDER BY dfm_id
""").show()

StatementMeta(, 192f92f6-78bb-4563-be75-a95dda87885b, -1, Cancelled, , Cancelled, True)